# Predict Seagrass at different times
* Designed to work year by year
* Pick a location and run

In [ ]:
import geopandas
import pathlib
import rioxarray
import numpy
import dask.distributed
import pandas
import rasterio

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

# Value to edit

In [ ]:
all_training_sites =  ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle",
                       "Robinsons", "Takamatua", "Purau", "Ihutai"]

In [ ]:
model_file = utils.get_model_file("trained_on_Duvauchelle_test_on_Robinsons")
prediction_sites = all_training_sites
year = 2025

# Cells to run
* Get lowtide satellite images by year and site
* Predict Seagrass across satellite images

In [ ]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()
max_cloud_cover = 0

### Save satellite
The low-tide no cloud images across sites for a whole year

In [ ]:
for prediction_site in all_training_sites:
    print(f"{prediction_site}: year {year}")

    satellite_file = data_path / "predictions" / "satellite_images" / f"{prediction_site}_{year}_sentinel-2.nc"
    site_polygon_file = data_path / "site_polygons" / f"{prediction_site}_polygon.gpkg"

    if not satellite_file.exists():
        print(f"\tSave satellite image of the site {prediction_site} around lowtide without cloud in the year {year}")
        for i in range(200):
            try:
                site_polygon = geopandas.read_file(site_polygon_file)
                satellite_data = sentinel2.get_low_tide_no_cloud_images_in_year(
                    geometry=site_polygon,
                    max_cloud_cover=max_cloud_cover,
                    year=year
                )
                if len(satellite_data['time']) == 0:
                    print("Warning: No satellite data without cloud. Ignore")
                else:
                    print("\t\tSave")
                    utils.write_netcdf_conventions_in_place(satellite_data)
                    satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                    satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                    utils.write_netcdf_conventions_in_place(satellite_data)
                    utils.save_netcdf(satellite_data, satellite_file)
                    break
            except Exception:
                print(f"\tIgnore error {Exception} and try again.")

### Predict values

In [ ]:
for prediction_site in all_training_sites:
    print(f"{prediction_site}: year {year}")
    prediction_file = data_path / "predictions" / "predictions" / f"{prediction_site}_{year}_prediction_{model_file.stem}.nc"
    polygon_file = data_path / "site_polygons" / f"{prediction_site}_polygon.gpkg"
    satellite_file = data_path / "predictions" / "satellite_images" / f"{prediction_site}_{year}_sentinel-2.nc"
    if not prediction_file.exists():
        predictions, satellite_data = training.predict_site(test_satellite_file=satellite_file, polygon_file=polygon_file, model_file=model_file)
        utils.save_netcdf(predictions, prediction_file)